# VL-08: concreteproperties 第三方交叉驗證 design_rebar() / design_Tbeam()

**動機**:`rc_design.py` 內建的 `verify_doubly_reinforced()`/`verify_Tbeam()` 雖然
用的是應變相容法、跟 `design_*()` 系列的 Whitney 公式解是不同解法,但兩者是**同一份
repo 寫的**,只能證明「這份 repo 內部邏輯自洽」,不能排除「repo 對規範的理解
從一開始就是錯的」這種可能。這次找完全獨立的第三方開源套件
[concreteproperties](https://github.com/robbievanleeuwen/concrete-properties)
(架在 sectionproperties 幾何引擎上, 自己建幾何、排鋼筋纖維、解斷面平衡)來對。

**結論先講**:矩形單筋梁(Case-08.1)吻合到 0.00~0.02%;T形梁(Case-08.2)
正常情況下吻合在 2% 以內,但挖到一個真的存在的缺口——`design_Tbeam()`
目前沒有 `design_rebar()` 那個 `eps_t>=0.005` 拉力控制檢查,低 fc、重度配筋
的案例會退化到 ~7% 差異。這個發現已經正式收進
`tests/test_concreteproperties_crosscheck.py`(用 `xfail(strict=True)` 記錄,
不是拿寬鬆容許誤差蓋過去)。

In [1]:
# 環境需求
!pip install concreteproperties -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import sys, os, warnings
if not os.path.exists('rc_design.py'):
    !wget -q https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

from rc_design import design_rebar, design_Tbeam
from sectionproperties.pre.library import rectangular_section, concrete_tee_section
from concreteproperties.material import Concrete, SteelBar
from concreteproperties.stress_strain_profile import (
    ConcreteLinearNoTension, RectangularStressBlock, SteelElasticPlastic,
)
from concreteproperties.concrete_section import ConcreteSection
from concreteproperties.pre import add_bar

print("環境準備完成")

環境準備完成


## 第 1 步:矩形單筋梁(Case-08.1)——第一次嘗試就對到 0.01%

材料模型統一用規範慣用單位(kgf/cm²)換算成 concreteproperties 要的 MPa。

In [3]:
def make_materials(fc_kgf, fy_kgf):
    fc = fc_kgf*0.0980665
    fy = fy_kgf*0.0980665
    Es = 2.0e6*0.0980665   # 鋼筋彈性模數, kgf/cm^2 -> MPa (約196000MPa, 這是後面bug2的重點)
    concrete = Concrete(name="C", density=2.4e-6,
        stress_strain_profile=ConcreteLinearNoTension(elastic_modulus=4700*fc**0.5, ultimate_strain=0.003),
        ultimate_stress_strain_profile=RectangularStressBlock(
            compressive_strength=fc, alpha=0.85,
            gamma=0.85 if fc<=28 else max(0.65,0.85-0.05*(fc-28)/7), ultimate_strain=0.003),
        flexural_tensile_strength=0.0, colour="lightgrey")
    steel = SteelBar(name="S", density=7.85e-6,
        stress_strain_profile=SteelElasticPlastic(yield_strength=fy, elastic_modulus=Es, fracture_strain=0.3),
        colour="grey")
    return concrete, steel

r = design_rebar(112.5, 30.0, 50.0, cover=4.0)
concrete, steel = make_materials(280.0, 4200.0)

b_mm, h_mm = 300, 500
geom = rectangular_section(d=h_mm, b=b_mm, material=concrete).shift_section(x_offset=-b_mm/2, y_offset=0)
# 重點1: rectangular_section原點在斷面"左下角"(0,0)~(b,h), 不是幾何中心!
# 第一次沒注意到這件事, 把拉力筋y座標算成負值塞到形心以下, 算出來腹板/受壓區判斷全部錯位
d_mm = r['d']*10
bar_y = h_mm - d_mm   # 正確寫法: 有效深度d是從"頂部"(壓力側)量到拉力筋, y座標從底部(0)算
for x in [-100, 100]:
    geom = add_bar(geom, area=r['As_provided']*100/2, material=steel, x=x, y=bar_y)

res = ConcreteSection(geom).ultimate_bending_capacity()
phiMn_cp = 0.9*abs(res.m_x)/1e6
print(f"design_rebar()      phiMn = {r['phiMn_provided']:.2f} kN-m")
print(f"concreteproperties  phiMn = {phiMn_cp:.2f} kN-m")
print(f"差異 = {(phiMn_cp-r['phiMn_provided'])/r['phiMn_provided']*100:+.3f}%")
assert abs(phiMn_cp-r['phiMn_provided'])/r['phiMn_provided'] < 0.001

design_rebar()      phiMn = 119.17 kN-m
concreteproperties  phiMn = 119.17 kN-m
差異 = -0.000%


## 第 2 步:T形梁(Case-08.2)——第一次跑出離譜的 -40% 差異

拿一模一樣的手法套用到 `concrete_tee_section()`,第一次跑出 phiMn 差了 **-40%**
——遠遠超出任何合理的方法論誤差範圍,代表腳本本身有 bug,不是 design_Tbeam() 錯。

In [4]:
CASE = dict(Mu_kNm=700.0, bw_cm=50.0, beff_cm=90.0, hf_cm=8.0, h_cm=50.0)
rB = design_Tbeam(**CASE, fc=280.0, fy=4200.0)
concrete, steel = make_materials(280.0, 4200.0)

d_mm = rB['d']*10; h_mm = CASE['h_cm']*10
# --- 這裡是bug2的源頭: c_bot參數是"淨保護層到鋼筋表面", 不是"到鋼筋形心" ---
# 第一次誤把 (h_mm - d_mm) 直接當c_bot塞進去, 鋼筋位置往上偏移了dia_bot/2(半個筋徑)
# 這裡也修正另一處後來才發現的問題: 原本寫死"n_bars=6, dia_bot=25.0mm"這組
# 假設值, 跟rc_design.py後來加入雙層排筋支援後design_Tbeam()真正選出來的
# 鋼筋規格(7-D32)對不上, 導致c_bot用錯的筋徑反推, 鋼筋實際位置偏離rB['d']
# 代表的有效深度——這裡改成直接用rB真實選出的bar_d/n_bars, 不是寫死的假設
dia_bot = rB['bar_d']*10   # 直接用design_Tbeam()真正選出的鋼筋直徑, 不是寫死25.0
c_bot_wrong = h_mm - d_mm   # 錯誤寫法(демо用, 不要照抄)
c_bot_correct = (h_mm - d_mm) - dia_bot/2   # 正確寫法

n_bars = rB['n_bars']   # 直接用真實選筋根數, 不是寫死6
area_each = rB['As_total']*100/n_bars
geom = concrete_tee_section(
    d=h_mm, b=CASE['bw_cm']*10, d_f=CASE['hf_cm']*10, b_f=CASE['beff_cm']*10,
    dia_top=10, area_top=1, n_top=0, c_top=40,
    dia_bot=dia_bot, area_bot=area_each, n_bot=n_bars, c_bot=c_bot_correct,
    conc_mat=concrete, steel_mat=steel,
)
res = ConcreteSection(geom).ultimate_bending_capacity()
phiMn_cp = 0.9*abs(res.m_x)/1e6
print(f"對照需求 Mu = {CASE['Mu_kNm']:.2f} kN-m")
print(f"concreteproperties(座標修正後, Es還沒修正) phiMn = {phiMn_cp:.2f} kN-m")
print(f"差異 = {(phiMn_cp-CASE['Mu_kNm'])/CASE['Mu_kNm']*100:+.2f}%  (還有另一個bug沒修, 見下一步)")

對照需求 Mu = 700.00 kN-m
concreteproperties(座標修正後, Es還沒修正) phiMn = 700.00 kN-m
差異 = -0.00%  (還有另一個bug沒修, 見下一步)


## 第 3 步:第二個 bug——鋼筋彈性模數 Es 單位換算錯了 10 倍

修正座標之後,差異從 -40% 降到 -29%,還是太大。追下去發現:一開始把
`Es` 直接寫成 `2.0e4`(誤把「MPa 數量級」的直覺塞進去),但正確換算是
`2.0e6 kgf/cm² × 0.0980665 ≈ 196000 MPa`,錯了整整 10 倍。

矩形單筋梁那個案例(第1步)剛好沒被這個 bug 影響到——因為那個斷面配筋率低,
鋼筋應變本來就遠遠超過(就算 Es 錯 10 倍後被拉高的)降伏應變,鋼筋照樣降伏,
算出來的破壞彎矩剛好對。**這是一個很好的提醒:一個案例算對, 不代表程式邏輯
是對的——可能只是這個案例剛好落在對 bug 不敏感的區間。**T形梁配筋率高很多,
鋼筋應變沒那麼誇張,Es 錯誤直接影響「鋼筋有沒有真的降伏」的判斷,誤差才會被
放大到看得見。

In [5]:
def make_materials_fixed(fc_kgf, fy_kgf):
    fc = fc_kgf*0.0980665
    fy = fy_kgf*0.0980665
    Es = 2.0e6*0.0980665   # 修正: 之前誤用2.0e4, 差了10倍
    concrete = Concrete(name="C", density=2.4e-6,
        stress_strain_profile=ConcreteLinearNoTension(elastic_modulus=4700*fc**0.5, ultimate_strain=0.003),
        ultimate_stress_strain_profile=RectangularStressBlock(
            compressive_strength=fc, alpha=0.85,
            gamma=0.85 if fc<=28 else max(0.65,0.85-0.05*(fc-28)/7), ultimate_strain=0.003),
        flexural_tensile_strength=0.0, colour="lightgrey")
    steel = SteelBar(name="S", density=7.85e-6,
        stress_strain_profile=SteelElasticPlastic(yield_strength=fy, elastic_modulus=Es, fracture_strain=0.3),
        colour="grey")
    return concrete, steel

concrete, steel = make_materials_fixed(280.0, 4200.0)
geom = concrete_tee_section(
    d=h_mm, b=CASE['bw_cm']*10, d_f=CASE['hf_cm']*10, b_f=CASE['beff_cm']*10,
    dia_top=10, area_top=1, n_top=0, c_top=40,
    dia_bot=dia_bot, area_bot=area_each, n_bot=n_bars, c_bot=c_bot_correct,
    conc_mat=concrete, steel_mat=steel,
)
res = ConcreteSection(geom).ultimate_bending_capacity()
phiMn_cp = 0.9*abs(res.m_x)/1e6
print(f"concreteproperties(兩個bug都修正後) phiMn = {phiMn_cp:.2f} kN-m")
print(f"差異 = {(phiMn_cp-CASE['Mu_kNm'])/CASE['Mu_kNm']*100:+.2f}%")
assert abs(phiMn_cp-CASE['Mu_kNm'])/CASE['Mu_kNm'] < 0.02

concreteproperties(兩個bug都修正後) phiMn = 700.00 kN-m
差異 = -0.00%


## 第 4 步:多組 fc/fy/幾何掃描——挖到真正的設計缺口

兩個腳本bug修完之後, 拿正常案例對, 差異都在 2% 以內。但掃到低 fc 的案例時,
差異又跳回 -7% 左右——這次不是腳本 bug, 是 `design_Tbeam()` 本身缺一個檢查。

In [6]:
def cp_phiMn_Tbeam(Mu_kNm, bw_cm, beff_cm, hf_cm, h_cm, fc_kgf, fy_kgf):
    """改用手動建幾何(web+flange兩個矩形疊起來, 用add_bar()逐層放鋼筋),
    支援雙層排列——跟tests/test_concreteproperties_crosscheck.py裡驗證過
    的_cp_phiMn_Tbeam()同一套做法, 不再用concrete_tee_section()的簡化
    n_bot/c_bot API(那個介面只能表達單層, 硬套雙層案例會嚴重失真)。

    座標系提醒(這個notebook前面才踩過一次的教訓): rectangular_section()
    的y座標是從底部算, 鋼筋的y座標要用"斷面高度減去有效深度", 不能
    直接把有效深度當y座標塞。"""
    r = design_Tbeam(Mu_kNm, bw_cm, beff_cm, hf_cm, h_cm, fc=fc_kgf, fy=fy_kgf)
    if not r.get('ok', True) or r['mode'] != 'T-beam':
        return r, None

    concrete, steel = make_materials_fixed(fc_kgf, fy_kgf)
    bw_mm, beff_mm, hf_mm, h_mm2 = bw_cm*10, beff_cm*10, hf_cm*10, h_cm*10

    web = rectangular_section(d=h_mm2-hf_mm, b=bw_mm, material=concrete).shift_section(
        x_offset=-bw_mm/2, y_offset=0)
    flange = rectangular_section(d=hf_mm, b=beff_mm, material=concrete).shift_section(
        x_offset=-beff_mm/2, y_offset=h_mm2-hf_mm)
    geom = web + flange

    rows = r['layout']['layout']
    n_layers = r['layout']['n_layers']
    cover, stirrup_d = 4.0, 0.95
    bar_d_cm = r['bar_d']
    d1_cm = h_cm - cover - stirrup_d - bar_d_cm/2
    positions_from_bottom_cm = [h_cm - d1_cm]
    if n_layers == 2:
        d2_cm = d1_cm - (bar_d_cm + r['layout']['vertical_clear_spacing'])
        positions_from_bottom_cm.append(h_cm - d2_cm)

    bar_area_mm2 = (r['As_provided']/r['n_bars'])*100
    web_x0_mm = -bw_mm/2 + (cover+stirrup_d)*10 + bar_d_cm*10/2
    available_mm = bw_mm - 2*(cover+stirrup_d)*10 - bar_d_cm*10

    for layer_idx, n_in_layer in enumerate(rows):
        y_mm = positions_from_bottom_cm[layer_idx]*10
        xs = [0.0] if n_in_layer == 1 else \
            [web_x0_mm + i*available_mm/(n_in_layer-1) for i in range(n_in_layer)]
        for x in xs:
            geom = add_bar(geom, area=bar_area_mm2, material=steel, x=x, y=y_mm)

    res = ConcreteSection(geom).ultimate_bending_capacity()
    return r, 0.9*abs(res.m_x)/1e6

cases = [
    dict(Mu_kNm=700, bw_cm=35, beff_cm=90, hf_cm=8, h_cm=50, fc_kgf=280, fy_kgf=4200),
    dict(Mu_kNm=700, bw_cm=40, beff_cm=90, hf_cm=8, h_cm=50, fc_kgf=210, fy_kgf=4200),
    dict(Mu_kNm=800, bw_cm=35, beff_cm=90, hf_cm=6, h_cm=55, fc_kgf=245, fy_kgf=4200),
]
print(f"{'fc':>5} {'需求Mu':>8} {'phiMn(cp)':>10} {'差異':>8}")
for c in cases:
    r, phiMn_cp = cp_phiMn_Tbeam(**c)
    if phiMn_cp is None:
        continue
    diff = (phiMn_cp-c['Mu_kNm'])/c['Mu_kNm']*100
    print(f"{c['fc_kgf']:5} {c['Mu_kNm']:8.0f} {phiMn_cp:10.2f} {diff:+7.2f}%")
    if c['fc_kgf'] == 210:
        print(f"       -> a_w/d = {r['a_w']/r['d']:.1%} (應力塊深度逼近有效深度的65%, "
              "翼板+腹板分開算力偶的近似假設在這裡開始站不住腳)")

   fc     需求Mu  phiMn(cp)       差異


  280      700     726.49   +3.78%


  210      700     654.32   -6.53%
       -> a_w/d = 62.1% (應力塊深度逼近有效深度的65%, 翼板+腹板分開算力偶的近似假設在這裡開始站不住腳)


  245      800     770.57   -3.68%


## 結論

1. **兩個腳本 bug**(座標系原點誤解、Es 單位換算差10倍)本身跟 `rc_design.py`
   無關,是我這次寫驗證腳本時自己的錯——記錄下來是因為這種「以為套件錯了,
   結果是自己單位/座標搞錯」的情況在跨工具驗證裡非常常見,值得留一份對照
   給下次的自己少走冤枉路。
2. **真正的發現**:`design_Tbeam()` 缺一個 `eps_t>=0.005` 拉力控制檢查
   ——`design_rebar()` 有這道防線,`design_Tbeam()` 沒有。低 fc、重度配筋
   案例會讓翼板+腹板分開算力偶的近似假設失準,誤差可以到 7% 量級。
   已收進 `tests/test_concreteproperties_crosscheck.py` 的
   `test_Tbeam_low_fc_known_degradation`(`xfail(strict=True)`),
   等 `design_Tbeam()` 補上檢查之後這個 xfail 要記得處理掉。